# Positional Encoding：从正弦到 RoPE 与 ALiBi

## 学习目标

1. 用置换等变性解释 attention 为何需要位置结构。
2. 从零实现正弦位置编码、RoPE、ALiBi 和 masked position_ids。
3. 理解绝对、相对、bias 三种注入路径及长度插值的取舍。
4. 能排查左右 padding、off-by-one、低精度角度和配置错配。

核心公式：$PE(p,2i)=\sin(p/10000^{2i/d})$，$PE(p,2i+1)=\cos(p/10000^{2i/d})$。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(11)

def stable_softmax(x, axis=-1):
    shifted = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(shifted)
    return e / e.sum(axis=axis, keepdims=True)

## 1. 无位置 attention 的置换等变性

若同时重排输入 token，$QK^\top$ 的行列同步重排，输出也只同步重排：$Attn(PX)=PAttn(X)$。下面用无投影的简化 self-attention 验证；这不包含 causal mask，因为 causal mask 本身已经注入顺序结构。

In [ ]:
def plain_attention(x):
    scores = x @ x.T / np.sqrt(x.shape[-1])
    return stable_softmax(scores) @ x

x = rng.normal(size=(4, 6))
perm = np.array([2, 0, 3, 1])
original = plain_attention(x)
permuted = plain_attention(x[perm])
print('max equivariance error =', np.max(np.abs(permuted - original[perm])))
assert np.allclose(permuted, original[perm])

## 2. 正弦位置编码

不同二维对使用几何分布频率：高频区分局部相邻，低频覆盖更长尺度。sin/cos 成对使位移可表示为二维旋转，并有 $u_p^\top u_q=\cos((p-q)\omega)$。

In [ ]:
def sinusoidal_positions(length, d_model, base=10000.0, offset=0):
    if d_model % 2:
        raise ValueError('示例要求 d_model 为偶数')
    positions = np.arange(offset, offset + length, dtype=np.float64)[:, None]
    inv_freq = base ** (-np.arange(0, d_model, 2, dtype=np.float64) / d_model)
    angles = positions * inv_freq[None, :]
    pe = np.empty((length, d_model), dtype=np.float64)
    pe[:, 0::2], pe[:, 1::2] = np.sin(angles), np.cos(angles)
    return pe

pe = sinusoidal_positions(6, 8)
print('PE.shape =', pe.shape)
print('position 0 =', pe[0])
print('相邻位置欧氏距离 =', np.linalg.norm(pe[1:] - pe[:-1], axis=1))

## 3. 验证单频率只依赖相对距离

二维位置向量 $u_p=(\sin p\omega,\cos p\omega)$。固定距离 $\Delta$，无论绝对起点如何，点积都应为 $\cos(\Delta\omega)$。

In [ ]:
omega, delta = 0.17, 5
def pair(position):
    return np.array([np.sin(position * omega), np.cos(position * omega)])

dots = [pair(p) @ pair(p + delta) for p in [0, 3, 20, 100]]
print('same-distance dot products =', dots)
print('expected =', np.cos(delta * omega))
assert np.allclose(dots, np.cos(delta * omega))

## 4. 从零实现 RoPE

RoPE 把每两个 Q/K 维度按位置旋转。正交旋转保持范数，并满足 $(R_mq)^\top(R_nk)=q^\top R_{n-m}k$。这里使用相邻维配对；真实 checkpoint 也可能用前后半维配对，布局不能混用。

In [ ]:
def apply_rope(vectors, positions, base=10000.0):
    vectors = np.asarray(vectors, dtype=np.float64)
    d = vectors.shape[-1]
    if d % 2:
        raise ValueError('rotary dimension 必须为偶数')
    inv_freq = base ** (-np.arange(0, d, 2, dtype=np.float64) / d)
    angles = np.asarray(positions, dtype=np.float64)[..., None] * inv_freq
    c, s = np.cos(angles), np.sin(angles)
    even, odd = vectors[..., 0::2], vectors[..., 1::2]
    out = np.empty_like(vectors)
    out[..., 0::2] = even * c - odd * s
    out[..., 1::2] = even * s + odd * c
    return out

q = rng.normal(size=(4, 8)); k = rng.normal(size=(4, 8))
pos = np.arange(4)
q_rot, k_rot = apply_rope(q, pos), apply_rope(k, pos)
print('max norm error =', np.max(np.abs(np.linalg.norm(q, axis=1) - np.linalg.norm(q_rot, axis=1))))

# 共同平移位置后，不加 mask 的 QK score 应保持。
score_a = q_rot @ k_rot.T
score_b = apply_rope(q, pos + 37) @ apply_rope(k, pos + 37).T
print('common-shift score error =', np.max(np.abs(score_a - score_b)))

## 5. ALiBi：在 logit 上加距离先验

因果 ALiBi 可写成 $s_{ij}=q_i^\top k_j/\sqrt d-m_h(i-j)$（$j\le i$）。不同 head 的斜率提供不同距离尺度。它没有位置表，但仍不降低 $L^2$ 计算。

In [ ]:
def causal_alibi_bias(length, slopes):
    i = np.arange(length)[:, None]
    j = np.arange(length)[None, :]
    distance = i - j
    visible = distance >= 0
    bias = -np.asarray(slopes)[:, None, None] * distance[None, :, :]
    return np.where(visible[None, :, :], bias, -np.inf)

bias = causal_alibi_bias(5, slopes=[0.25, 0.03125])
print('bias.shape [H,L,L] =', bias.shape)
print('head 0:\n', bias[0])

## 6. Padding 与 position_ids

Left padding 后不能总用数组索引。常见做法是 mask 的累积和减一，让每条有效序列从 0 开始；PAD 填安全 id，再由 attention mask 屏蔽。Position id、attention mask、label mask 是三个独立协议。

In [ ]:
def positions_from_mask(mask):
    mask = np.asarray(mask, dtype=bool)
    positions = np.cumsum(mask, axis=1) - 1
    return np.where(mask, positions, 0).astype(np.int64)

left_mask = np.array([[0, 0, 1, 1, 1], [0, 1, 1, 1, 1]])
right_mask = np.array([[1, 1, 1, 0, 0], [1, 1, 1, 1, 0]])
print('left positions:\n', positions_from_mask(left_mask))
print('right positions:\n', positions_from_mask(right_mask))

## 7. 位置插值与长上下文

若原长度 $L$ 扩为 $sL$，线性位置插值使用 $p'=p/s$，把新位置压回原相位范围；代价是相邻位置角度差也缩小。公式可计算任意位置不等于模型有真实长程能力，仍需长依赖数据和多距离评估。

In [ ]:
original_length, target_length = 8, 32
factor = target_length / original_length
new_positions = np.arange(target_length)
interpolated = new_positions / factor
print('new last position =', new_positions[-1])
print('mapped last position =', interpolated[-1])
print('adjacent mapped gap =', interpolated[1] - interpolated[0])

# 大 position 的角度构造应在 float32/float64 完成，不能先转低精度。
large = np.array([100000, 100001], dtype=np.float64)
print('float64 positions distinguishable:', large[1] - large[0])

## 工程变体、常见坑与练习

- 可学习绝对表灵活但有最大行；正弦无参数；T5 bias 按距离 bucket；RoPE 旋转 Q/K；ALiBi 加线性 logit bias。
- RoPE 的 base、rotary dimension、相邻/半区配对和 scaling 都属于 checkpoint 协议，shape 相同也可能静默错。
- Packed 独立样本必须同时做 block mask、位置重置和 label 边界；只重置 position 不能阻止跨样本读取。
- 评估使用“长度 × 证据位置 × 距离 × 干扰”矩阵，同时保留短上下文回归。

练习：①把 RoPE 改成只旋转前 4 维；②验证只平移 Q 而不平移 K 时 score 会变化；③为双向 relative bias 编写有符号 bucket；④构造 full-forward 与逐步 position_ids 的一致性测试。

面试主线：置换等变性 → 注入点（输入/QK/logit）→ 相对位移公式 → 外推不等于能力 → padding、精度和配置校验。